# Environment Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "orb-models==0.5.5", "ase>=3.24", "numpy>=1.26", "scipy>=1.15", "tqdm>=4.66", "cached_path>=1.6.7", "dm-tree==0.1.8", "pandas>=2.2", "h5py>=3.11", "matplotlib>=3.9"])


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "cudf", "cuml", "libcugraph-cu12", "pylibcugraph-cu12", "pylibraft-cu12", "libraft-cu12", "rmm-cu12"])


In [ ]:
# Data is expected under ../data/raw (see data/README.md)


# Params

In [ ]:
# Model size choices: tiny -> orb-d3-xs-v2, small -> orb-d3-sm-v2, full -> orb-v2
from pathlib import Path

MODEL_SIZE = "tiny"

# Sampling / data (set your dataset paths here)
RAW_DEEPMD_DIRS = [
    "../data/raw/beta-Li3PS4",
    "../data/raw/gamma-Li3PS4",
]
# If you already have XYZ files, set RAW_DEEPMD_DIRS = [] and fill RAW_DATASETS.
RAW_DATASETS = [f"../data/raw/{Path(path).name}.xyz" for path in RAW_DEEPMD_DIRS]

SAMPLE_SIZE = 1250  # frames per dataset for prepare_datasets

# Names inferred from file stems (override DATASET_NAMES if desired)
DATASET_NAMES = [Path(path).stem for path in RAW_DATASETS]
COMBINED_NAME = "_".join(DATASET_NAMES)

# Training hyperparams (defaults)
DEFAULT_EPOCHS = 100
DEFAULT_LR = 5e-5
DEFAULT_WEIGHT_DECAY = 3e-4
BATCH_SIZE = 16
VAL_FRAC = 0.1
TEST_FRAC = 0.1
SPLIT_SEED = 42
TRAIN_SEED = 42
CSV_SUFFIX = f"_seed{TRAIN_SEED}"
DATA_UNITS = "eV"  # dataset units; converted to eV for training/eval

# Per-model base training overrides (set only what differs)
BASE_TRAIN_OVERRIDES = {
    # Example:
    "beta-Li3PS4": {"epochs": 150, "lr": 5e-4},
    "gamma-Li3PS4": {"epochs": 150, "lr": 5e-4},
    "beta-Li3PS4_gamma-Li3PS4": {"epochs": 150, "lr": 5e-4},
}

# Force-head fine-tune (swap embeddings)
FT_SAMPLES = 1024
FT_EPOCHS = 100
FT_LR = 5e-3
FT_SCHED = "none"
FT_SCHED_GAMMA = 0.5
FT_SCHED_PATIENCE = 4

# Fine-tune validation controls
FT_VAL_EVERY = 1   # 0 disables per-epoch val checks
FT_VAL_LIMIT = 0   # 0 uses full val split
FT_NO_VAL = False  # True disables val loss completely

# Stacks+head fine-tune
STACKS = 1          # 1/2/3/4
STACKS_LR = 5e-4    # backbone LR
STACKS_EPOCHS = 100
STACKS_SAMPLES = 512
STACKS_SCHED = "none"
STACKS_VAL_FRAC = None  # override val fraction for stacks fine-tune; set 0 to disable

# Toggle AMP for speed on GPU (doesn't work correctly for now)
USE_AMP = False 



In [ ]:
import os, subprocess, shlex, sys, time, re

os.environ.setdefault("PYTHONUNBUFFERED", "1")

def run(cmd):
    cmd_list = shlex.split(cmd) if isinstance(cmd, str) else list(cmd)
    print("\n>>>", " ".join(cmd_list))
    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        sys.stdout.write(line)
    ret = proc.wait()
    if ret:
        raise SystemExit(ret)

def run_capture(cmd):
    cmd_list = shlex.split(cmd) if isinstance(cmd, str) else list(cmd)
    print("\n>>>", " ".join(cmd_list))
    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    output = []
    for line in proc.stdout:
        sys.stdout.write(line)
        output.append(line)
    ret = proc.wait()
    if ret:
        raise SystemExit(ret)
    return "".join(output)

def parse_logged_time(text, pattern):
    match = re.search(pattern, text, re.M)
    if match:
        return float(match.group(1))
    return None



# Prepare Datasets

In [ ]:
import time, json
from pathlib import Path

if RAW_DEEPMD_DIRS:
    cmd = [
        "python", "-u", "../scripts/convert_deepmd_raw_to_xyz.py",
        "--output-dir", "../data/raw",
    ]
    for raw_dir in RAW_DEEPMD_DIRS:
        name = Path(raw_dir).name
        cmd += ["--dataset", f"{name}={raw_dir}"]
    run(cmd)

cmd = [
    "python", "-u", "../scripts/prepare_datasets.py",
    "--combined-name", COMBINED_NAME,
    "--sample-size", str(SAMPLE_SIZE),
    "--output-dir", "../data/prepared",
]
for name, path in zip(DATASET_NAMES, RAW_DATASETS):
    cmd += ["--dataset", f"{name}={path}"]
start = time.time()
run(cmd)
print(f"prep_time_sec={int(time.time() - start)}")

summary_path = Path("../data/prepared/sampling_summary.json")
summary = json.loads(summary_path.read_text())

def dataset_entry(name: str):
    return next(entry for entry in summary["datasets"] if entry["name"] == name)

DATASET_DBS = {name: Path(dataset_entry(name)["db_path"]) for name in DATASET_NAMES}
COMBINED_DB = Path(dataset_entry(COMBINED_NAME)["db_path"])
for name in DATASET_NAMES:
    print(f"Dataset: {name} -> {DATASET_DBS[name]}")
print(f"Combined: {COMBINED_NAME} -> {COMBINED_DB}")



# Train ethanol/malonaldehyde/combined (timed)

In [ ]:
import time

TRAIN_TIMES = {}


def resolve_train_setting(name: str, key: str, default):
    overrides = BASE_TRAIN_OVERRIDES.get(name, {})
    if key not in overrides and name != COMBINED_NAME:
        overrides = BASE_TRAIN_OVERRIDES.get("combined", {})
    return overrides.get(key, default)


def train(name, db, out):
    epochs = resolve_train_setting(name, "epochs", DEFAULT_EPOCHS)
    lr = resolve_train_setting(name, "lr", DEFAULT_LR)
    weight_decay = resolve_train_setting(name, "weight_decay", DEFAULT_WEIGHT_DECAY)

    start = time.time()
    output = run_capture([
        "python", "-u", "../scripts/train_orb.py",
        "--dataset-name", name,
        "--db-path", str(db),
        "--output-dir", out,
        "--data-units", DATA_UNITS,
        "--model-size", MODEL_SIZE,
        "--epochs", str(epochs),
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(lr),
        "--weight-decay", str(weight_decay),
        "--no-scheduler",
        "--val-fraction", str(VAL_FRAC),
        "--test-fraction", str(TEST_FRAC),
        "--seed", str(TRAIN_SEED),
        "--split-seed", str(SPLIT_SEED),
        "--force-only",
    ])
    logged = parse_logged_time(
        output,
        r"Total training time:\s*([0-9.]+)s(?:\s*\(Actual\))?",
    )
    elapsed = logged if logged is not None else (time.time() - start)
    TRAIN_TIMES[name] = elapsed
    print(f"{name}_train_time_sec={elapsed:.2f}")

for name in DATASET_NAMES:
    train(name, DATASET_DBS[name], f"../models/trained/{name}_{MODEL_SIZE}")
train(COMBINED_NAME, COMBINED_DB, f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}")



# Evaluation of Trained Models

In [ ]:
def eval_model(name):
    cfg = f"../models/trained/{name}_{MODEL_SIZE}/{name}_config.json"
    ckpt = f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt"
    run([
        "python", "-u", "../scripts/evaluate_model.py",
        "--config", cfg,
        "--checkpoint", ckpt,
        "--split", "test",
        "--batch-size", "8",
    ])

for name in DATASET_NAMES + [COMBINED_NAME]:
    eval_model(name)



# Merge models (mean, closed-form, individual-fixed)

In [ ]:
import os, time
os.makedirs("../models/merged", exist_ok=True)
CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"

CKPTS = {name: f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt" for name in DATASET_NAMES}

MERGE_TIMES = {}

start = time.time()
cmd = [
    "python", "-u", "../scripts/merge_mean.py",
    "--config", CONFIG,
    "--output", "../models/merged/merged_mean.ckpt", "--keep-metadata",
]
for ckpt in CKPTS.values():
    cmd += ["--teacher", ckpt]
run(cmd)
MERGE_TIMES["merged_mean"] = time.time() - start

start = time.time()
cmd = [
    "python", "-u", "../scripts/merge_closed_form.py",
    "--config", CONFIG,
    "--dataset", str(COMBINED_DB),
    "--output", "../models/merged/merged_closed_form.ckpt",
    "--regularization", "1e-6", "--batch-size", "64",
]
for ckpt in CKPTS.values():
    cmd += ["--teacher", ckpt]
run(cmd)
MERGE_TIMES["merged_closed_form"] = time.time() - start

start = time.time()
cmd = [
    "python", "-u", "../scripts/merge_closed_form_individual_fixed.py",
    "--output", "../models/merged/merged_closed_form_individual_new.ckpt",
    "--regularization", "1e-6", "--batch-size", "64","--limit","500",
]
for name in DATASET_NAMES:
    cfg = f"../models/trained/{name}_{MODEL_SIZE}/{name}_config.json"
    ckpt = CKPTS[name]
    db = DATASET_DBS[name]
    cmd += ["--teacher", f"{name}|{cfg}|{ckpt}|{db}"]
merge_output = run_capture(cmd)
logged = parse_logged_time(
    merge_output,
    r"Closed-form individual merge compute time:\s*([0-9.]+)s",
)
MERGE_TIMES["merged_closed_form_individual"] = (
    logged if logged is not None else (time.time() - start)
)



# Evaluation of Merged Models

In [ ]:
CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"
DATA = str(COMBINED_DB)
OUTDIR = "../models/merged"

def eval_ckpt(stem):
    ckpt = f"{OUTDIR}/{stem}.ckpt"
    run([
        "python", "-u", "../scripts/evaluate_model.py",
        "--config", CONFIG,
        "--checkpoint", ckpt,
        "--dataset", DATA,
        "--split", "test",
        "--batch-size", "8",
    ])

eval_ckpt("merged_mean")
eval_ckpt("merged_closed_form")
eval_ckpt("merged_closed_form_individual_new")


# Switch-embedding Evaluation of Individual-Closed-Form-Merge

In [ ]:
cmd = [
    "python", "-u", "../scripts/evaluate_switch_embeddings.py",
    "--config", f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json",
    "--checkpoint", "../models/merged/merged_closed_form_individual_new.ckpt",
    "--dataset", str(COMBINED_DB),
]
for name in DATASET_NAMES:
    ckpt = f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt"
    cmd += ["--source-checkpoint", f"{name}={ckpt}"]
cmd += ["--split", "test", "--batch-size", "1", "--save", "../results/merged_closed_form_individual_switch_eval.txt"]
run(cmd)



# Force-head fine-tune (swap embeddings, timed)

In [ ]:
import time, os
from pathlib import Path
os.makedirs("logs", exist_ok=True)
OUT = f"../models/merged/merged_closed_form_individual_new_ft_s{FT_SAMPLES}_ep{FT_EPOCHS}_lr{FT_LR}_sched-{FT_SCHED}.ckpt"
LOG = f"../logs/{Path(OUT).stem}.log"
start = time.time()
cmd = [
    "python", "-u", "../scripts/force_head_ft/switch_finetune_embeddings.py",
    "--config", f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json",
    "--checkpoint", "../models/merged/merged_closed_form_individual_new.ckpt",
    "--dataset", str(COMBINED_DB),
]
for name in DATASET_NAMES:
    ckpt = f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt"
    cmd += ["--source-checkpoint", f"{name}={ckpt}"]
cmd += [
    "--ft-sample-graphs", str(FT_SAMPLES),
    "--ft-epochs", str(FT_EPOCHS),
    "--ft-lr", str(FT_LR),
    "--ft-scheduler", FT_SCHED,
    "--ft-scheduler-gamma", str(FT_SCHED_GAMMA),
    "--ft-scheduler-plateau-patience", str(FT_SCHED_PATIENCE),
    "--ft-output", OUT,
    "--ft-log-path", LOG,
    "--val-every", str(FT_VAL_EVERY),
    "--val-limit", str(FT_VAL_LIMIT),
    "--ft-cache-head-inputs",
]
if FT_NO_VAL:
    cmd.append("--no-val")
if USE_AMP:
    cmd.append("--amp")
output = run_capture(cmd)
logged = parse_logged_time(
    output,
    r"Fine-tuning duration:\s*([0-9.]+)\s*s",
)
FORCE_HEAD_FT_TIME_SEC = logged if logged is not None else (time.time() - start)
print(f"force_head_ft_time_sec={FORCE_HEAD_FT_TIME_SEC:.2f}")



# Evaluation of Force Head Finetuned Model

In [ ]:
from pathlib import Path
CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"
DATA = str(COMBINED_DB)
CKPTS = {name: f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt" for name in DATASET_NAMES}
CKPT = f"../models/merged/merged_closed_form_individual_new_ft_s{FT_SAMPLES}_ep{FT_EPOCHS}_lr{FT_LR}_sched-{FT_SCHED}.ckpt"
SAVE = f"../results/{Path(CKPT).stem}_switch_eval.txt"
cmd = [
    "python", "-u", "../scripts/evaluate_switch_embeddings.py",
    "--config", CONFIG,
    "--checkpoint", CKPT,
    "--dataset", DATA,
]
for name, ckpt in CKPTS.items():
    cmd += ["--source-checkpoint", f"{name}={ckpt}"]
cmd += ["--split", "test", "--batch-size", "1", "--save", SAVE]
run(cmd)



In [ ]:
import csv, re
from pathlib import Path


def _parse_key(text: str, key: str):
    match = re.search(rf"^\s*{re.escape(key)}:\s*([0-9.eE+-]+)\s*$", text, re.M)
    if match:
        return float(match.group(1))
    return None


def parse_force_metrics(path: Path):
    if not path.exists():
        return None, None
    text = path.read_text()
    match = re.search(r"Forces -> MAE: ([0-9.eE+-]+), RMSE: ([0-9.eE+-]+)", text)
    if match:
        return float(match.group(1)), float(match.group(2))
    mae = _parse_key(text, "raw_forces_mae") or _parse_key(text, "forces_mae") or _parse_key(text, "force_mae")
    rmse = _parse_key(text, "raw_forces_rmse") or _parse_key(text, "forces_rmse") or _parse_key(text, "force_rmse")
    return mae, rmse


def format_time(value):
    if value is None:
        return ""
    try:
        return f"{float(value):.2f}"
    except (TypeError, ValueError):
        return str(value)


def write_summary_row(writer, name: str, eval_path: Path, time_sec):
    mae, rmse = parse_force_metrics(eval_path)
    writer.writerow([
        name,
        "" if mae is None else f"{mae:.6f}",
        "" if rmse is None else f"{rmse:.6f}",
        format_time(time_sec),
    ])

train_times = globals().get("TRAIN_TIMES", {})
merge_times = globals().get("MERGE_TIMES", {})
ft_time = globals().get("FORCE_HEAD_FT_TIME_SEC")

rows = []
output_path = Path(f"experiment_summary{CSV_SUFFIX}.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["name", "force_mae", "force_rmse", "time_sec"])

    for name in DATASET_NAMES + [COMBINED_NAME]:
        eval_path = Path("results") / f"{name}_best_eval.txt"
        write_summary_row(writer, name, eval_path, train_times.get(name))

    write_summary_row(
        writer,
        "merged_mean",
        Path("results") / "merged_mean_eval.txt",
        merge_times.get("merged_mean"),
    )
    write_summary_row(
        writer,
        "merged_closed_form",
        Path("results") / "merged_closed_form_eval.txt",
        merge_times.get("merged_closed_form"),
    )
    write_summary_row(
        writer,
        "merged_closed_form_individual",
        Path("results") / "merged_closed_form_individual_new_eval.txt",
        merge_times.get("merged_closed_form_individual"),
    )
    write_summary_row(
        writer,
        "merged_closed_form_individual_switch",
        Path("results") / "merged_closed_form_individual_switch_eval.txt",
        None,
    )

    ft_ckpt = f"../models/merged/merged_closed_form_individual_new_ft_s{FT_SAMPLES}_ep{FT_EPOCHS}_lr{FT_LR}_sched-{FT_SCHED}.ckpt"
    ft_eval = Path("results") / f"{Path(ft_ckpt).stem}_switch_eval.txt"
    write_summary_row(writer, "force_head_finetune", ft_eval, ft_time)

print(f"Wrote {output_path}")



# Stacks + force-head fine-tune (timed)

In [ ]:
import time
from pathlib import Path
OUT = f"../models/merged/merged_closed_form_individual_new_ftstacks{STACKS}_s{STACKS_SAMPLES}_ep{STACKS_EPOCHS}_lr{FT_LR}_sched-{STACKS_SCHED}.ckpt"
LOG = f"../logs/{Path(OUT).stem}.log"
cmd = [
    "python", "-u", "../scripts/force_head_ft/switch_emb_ft_stacks.py",
    "--config", f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json",
    "--checkpoint", "../models/merged/merged_closed_form_individual_new.ckpt",
    "--dataset", str(COMBINED_DB),
]
for name in DATASET_NAMES:
    ckpt = f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt"
    cmd += ["--source-checkpoint", f"{name}={ckpt}"]
cmd += [
    "--ft-unfreeze-stacks", str(STACKS),
    "--ft-sample-graphs", str(STACKS_SAMPLES),
    "--ft-epochs", str(STACKS_EPOCHS),
    "--ft-lr", str(FT_LR),
    "--ft-backbone-lr", str(STACKS_LR),
    "--ft-scheduler", STACKS_SCHED,
    "--ft-scheduler-gamma", "0.5",
    "--ft-scheduler-plateau-patience", "4",
    "--ft-output", OUT,
    "--ft-log-path", LOG,
    "--val-every", str(FT_VAL_EVERY),
    "--val-limit", str(FT_VAL_LIMIT),
]
if STACKS_VAL_FRAC is not None:
    cmd += ["--ft-val-fraction", str(STACKS_VAL_FRAC)]
if FT_NO_VAL:
    cmd.append("--no-val")
if USE_AMP:
    cmd.append("--amp")
start = time.time()
output = run_capture(cmd)
logged = parse_logged_time(
    output,
    r"Fine-tuning duration:\s*([0-9.]+)\s*s",
)
elapsed = logged if logged is not None else (time.time() - start)
print(f"stacks_ft_time_sec={elapsed:.2f}")



# Evaluate all key models (forces MAE/RMSE saved per model)

In [ ]:
from pathlib import Path
CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"
DATA = str(COMBINED_DB)
CKPTS = {name: f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt" for name in DATASET_NAMES}
CKPT = f"../models/merged/merged_closed_form_individual_new_ftstacks{STACKS}_s{STACKS_SAMPLES}_ep{STACKS_EPOCHS}_lr{FT_LR}_sched-{STACKS_SCHED}.ckpt"
stem = Path(CKPT).stem
cmd = [
    "python", "-u", "../scripts/evaluate_switch_embeddings.py",
    "--config", CONFIG,
    "--checkpoint", CKPT,
    "--dataset", DATA,
]
for name, ckpt in CKPTS.items():
    cmd += ["--source-checkpoint", f"{name}={ckpt}"]
cmd += ["--split", "test", "--batch-size", "1", "--save", f"../results/{stem}_switch_eval.txt"]
run(cmd)



# All Stacks Tuning (select best LR on val, eval on test)


In [ ]:
import os, time, json, re, torch
from pathlib import Path
from collections import defaultdict

CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"
DATA = str(COMBINED_DB)
CKPTS = {name: f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt" for name in DATASET_NAMES}
SOURCE_ARGS = []
for name, ckpt in CKPTS.items():
    SOURCE_ARGS += ["--source-checkpoint", f"{name}={ckpt}"]

os.makedirs("../models/merged", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("results", exist_ok=True)

# (stacks, samples, epochs, lr)
COMBOS = [
  (3,125,15,"1e-4"),(3,125,15,"5e-4"),(3,125,15,"1e-3"),(3,125,15,"5e-3"),
  (3,125,30,"1e-4"),(3,125,30,"5e-4"),(3,125,30,"1e-3"),(3,125,30,"5e-3"),
  (3,125,50,"1e-4"),(3,125,50,"5e-4"),(3,125,50,"1e-3"),(3,125,50,"5e-3"),
  (3,125,75,"1e-4"),(3,125,75,"5e-4"),(3,125,75,"1e-3"),(3,125,75,"5e-3"),
  (3,250,15,"1e-4"),(3,250,15,"5e-4"),(3,250,15,"1e-3"),(3,250,15,"5e-3"),
  (3,250,30,"1e-4"),(3,250,30,"5e-4"),(3,250,30,"1e-3"),(3,250,30,"5e-3"),
  (3,250,50,"1e-4"),(3,250,50,"5e-4"),(3,250,50,"1e-3"),(3,250,50,"5e-3"),
  (3,250,75,"1e-4"),(3,250,75,"5e-4"),(3,250,75,"1e-3"),(3,250,75,"5e-3"),
  (3,500,15,"1e-4"),(3,500,15,"5e-4"),(3,500,15,"1e-3"),(3,500,15,"5e-3"),
  (3,500,30,"1e-4"),(3,500,30,"5e-4"),(3,500,30,"1e-3"),(3,500,30,"5e-3"),
  (3,500,50,"1e-4"),(3,500,50,"5e-4"),(3,500,50,"1e-3"),(3,500,50,"5e-3"),
  (3,500,75,"1e-4"),(3,500,75,"5e-4"),(3,500,75,"1e-3"),(3,500,75,"5e-3"),
  (3,750,15,"1e-4"),(3,750,15,"5e-4"),(3,750,15,"1e-3"),(3,750,15,"5e-3"),
  (3,750,30,"1e-4"),(3,750,30,"5e-4"),(3,750,30,"1e-3"),(3,750,30,"5e-3"),
  (3,750,50,"1e-4"),(3,750,50,"5e-4"),(3,750,50,"1e-3"),(3,750,50,"5e-3"),
  (3,750,75,"1e-4"),(3,750,75,"5e-4"),(3,750,75,"1e-3"),(3,750,75,"5e-3"),
]


def parse_metric(path: Path, key: str) -> float | None:
    text = path.read_text()
    pattern = rf"^\s*{re.escape(key)}:\s*([0-9.eE+-]+)\s*$"
    match = re.search(pattern, text, re.M)
    if match:
        return float(match.group(1))
    return None


def parse_val_loss(path: Path) -> float:
    loss = parse_metric(path, "loss")
    if loss is not None:
        return loss
    for key in ("raw_forces_mae", "forces_mae", "force_mae"):
        loss = parse_metric(path, key)
        if loss is not None:
            return loss
    raise ValueError(f"No loss metric found in {path}")


def read_best_from_ckpt(path: Path) -> tuple[float | None, int | None]:
    ckpt = torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict):
        best_val = ckpt.get("best_val_loss")
        best_epoch = ckpt.get("best_epoch")
        if best_val is not None:
            return float(best_val), int(best_epoch) if best_epoch is not None else None
    return None, None


by_group: dict[tuple[int, int, int], list[str]] = defaultdict(list)
for stacks, samples, epochs, lr in COMBOS:
    by_group[(stacks, samples, epochs)].append(lr)

summary = {
    "groups": [],
    "best": [],
}

for (stacks, samples, epochs), lrs in sorted(by_group.items()):
    best = None
    for lr in lrs:
        out = (
            f"../models/merged/merged_closed_form_individual_new_"
            f"ftstacks{stacks}_s{samples}_ep{epochs}_lr{lr}_sched-{STACKS_SCHED}.ckpt"
        )
        log = f"../logs/{Path(out).stem}.log"
        cmd = [
            "python", "-u", "../scripts/force_head_ft/switch_emb_ft_stacks.py",
            "--config", CONFIG,
            "--checkpoint", "../models/merged/merged_closed_form_individual_new.ckpt",
            "--dataset", DATA,
            "--ft-unfreeze-stacks", str(stacks),
            "--ft-sample-graphs", str(samples),
            "--ft-epochs", str(epochs),
            "--ft-lr", str(lr),
            "--ft-backbone-lr", str(lr),
            "--ft-scheduler", STACKS_SCHED,
            "--ft-scheduler-gamma", "0.5",
            "--ft-scheduler-plateau-patience", "4",
            "--ft-output", out,
            "--ft-log-path", log,
            "--val-every", str(FT_VAL_EVERY),
            "--val-limit", str(FT_VAL_LIMIT),
        ]
        cmd += SOURCE_ARGS
        if STACKS_VAL_FRAC is not None:
            cmd += ["--ft-val-fraction", str(STACKS_VAL_FRAC)]
        if FT_NO_VAL:
            cmd.append("--no-val")
        if USE_AMP:
            cmd.append("--amp")
        start = time.time()
        output = run_capture(cmd)
        logged = parse_logged_time(output, r"Fine-tuning duration:\s*([0-9.]+)\s*s")
        ft_time_sec = logged if logged is not None else (time.time() - start)
        print(f"stacks_ft_time_sec[{stacks},{samples},{lr}]={ft_time_sec:.2f}")

        val_path = Path("results") / f"{Path(out).stem}_switch_eval_val.txt"
        cmd = [
            "python", "-u", "../scripts/evaluate_switch_embeddings.py",
            "--config", CONFIG,
            "--checkpoint", out,
            "--dataset", DATA,
            "--split", "val",
            "--batch-size", "16",
            "--save", str(val_path),
        ]
        cmd += SOURCE_ARGS
        run(cmd)
        best_val_loss, best_epoch = read_best_from_ckpt(Path(out))
        val_loss = best_val_loss if best_val_loss is not None else parse_val_loss(val_path)
        entry = {
            "stacks": stacks,
            "samples": samples,
            "epochs": epochs,
            "lr": lr,
            "ckpt": out,
            "val_loss": val_loss,
            "best_epoch": best_epoch,
            "val_eval": str(val_path),
            "fine_tune_time_sec": ft_time_sec,
        }
        summary["groups"].append(entry)
        if best is None or val_loss < best["val_loss"]:
            best = entry

    if best is None:
        raise RuntimeError(f"No runs completed for stacks={stacks}, samples={samples}, epochs={epochs}")

    test_path = Path("results") / f"{Path(best['ckpt']).stem}_switch_eval_test.txt"
    cmd = [
        "python", "-u", "../scripts/evaluate_switch_embeddings.py",
        "--config", CONFIG,
        "--checkpoint", best["ckpt"],
        "--dataset", DATA,
        "--split", "test",
        "--batch-size", "1",
        "--save", str(test_path),
    ]
    cmd += SOURCE_ARGS
    run(cmd)
    best = {**best, "test_eval": str(test_path)}
    summary["best"].append(best)
    print(
        f"Best lr for stacks={stacks}, samples={samples}, epochs={epochs}: "
        f"lr={best['lr']} val_loss={best['val_loss']:.6f}"
    )

summary_path = Path("results") / "stacks_lr_selection.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Saved selection summary to {summary_path}")


In [ ]:
import csv, json, re
from pathlib import Path

summary_path = Path("../results/stacks_lr_selection.json")
if not summary_path.exists():
    raise FileNotFoundError(summary_path)
summary = json.loads(summary_path.read_text())


def parse_force_metrics(path: Path):
    text = path.read_text()
    def parse_key(key: str):
        match = re.search(rf"^\s*{re.escape(key)}:\s*([0-9.eE+-]+)\s*$", text, re.M)
        if match:
            return float(match.group(1))
        return None
    mae = parse_key("raw_forces_mae") or parse_key("forces_mae") or parse_key("force_mae")
    rmse = parse_key("raw_forces_rmse") or parse_key("forces_rmse") or parse_key("force_rmse")
    return mae, rmse


def format_time(value):
    if value is None:
        return ""
    try:
        return f"{float(value):.2f}"
    except (TypeError, ValueError):
        return str(value)

print("Best LR per (stack, limit, epoch):")
for entry in summary.get("best", []):
    print(
        f"  stack={entry['stacks']} limit={entry['samples']} epoch={entry['epochs']} "
        f"lr={entry['lr']} val_loss={entry['val_loss']:.6f} test_eval={entry['test_eval']}"
    )

csv_path = Path(f"stack_hyperparam_summary{CSV_SUFFIX}.csv")
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["stack", "limit", "epoch", "fine_tuning_time_sec", "best_lr", "force_mae", "force_rmse"])
    for entry in summary.get("best", []):
        test_eval = Path(entry["test_eval"])
        mae, rmse = parse_force_metrics(test_eval)
        writer.writerow([
            entry["stacks"],
            entry["samples"],
            entry["epochs"],
            format_time(entry.get("fine_tune_time_sec")),
            entry["lr"],
            "" if mae is None else f"{mae:.6f}",
            "" if rmse is None else f"{rmse:.6f}",
        ])

print(f"Wrote {csv_path}")

